In [15]:
import json
import pandas as pd
from pathlib import Path


# Resolve paths from either the project root or the scripts folder.
PROJECT_ROOT = next(
    (folder for folder in (Path.cwd(), *Path.cwd().parents)
     if (folder / "json_data_file").is_dir() and (folder / "scripts").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Open this notebook from the Nordic Energy project folder.")
RAW_DATA_PATH = PROJECT_ROOT / "json_data_file" / "prices.json"

# Display prices with 2 decimal places without changing the original values
pd.options.display.float_format = "{:.2f}".format


In [16]:
# Load the JSON data from the project folder
with RAW_DATA_PATH.open("r", encoding="utf-8") as file:
    data = json.load(file)

# Convert the JSON records into a Pandas DataFrame
df = pd.DataFrame(data["records"])

# Check the number of rows and columns
print(df.shape)

# Display the first 5 rows with prices rounded to 2 decimal places
display(
    df.head().style.format({
        "DayAheadPriceEUR": "{:.2f}",
        "DayAheadPriceDKK": "{:.2f}"
    })
)

(1344, 5)


,TimeUTC,TimeDK,PriceArea,DayAheadPriceEUR,DayAheadPriceDKK
0,2026-10-31T23:00:00,2026-11-01T00:00:00,DK1,257.39,1923.60
1,2026-10-31T23:00:00,2026-11-01T00:00:00,DK2,257.45,1924.05
2,2026-10-31T23:15:00,2026-11-01T00:15:00,DK1,251.04,1876.15
3,2026-10-31T23:15:00,2026-11-01T00:15:00,DK2,251.12,1876.75
4,2026-10-31T23:30:00,2026-11-01T00:30:00,DK1,239.71,1791.47


In [17]:
# Check the data types and missing values in the DataFrame
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1344 entries, 0 to 1343
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   TimeUTC           1344 non-null   object 
 1   TimeDK            1344 non-null   object 
 2   PriceArea         1344 non-null   object 
 3   DayAheadPriceEUR  1344 non-null   float64
 4   DayAheadPriceDKK  1344 non-null   float64
dtypes: float64(2), object(3)
memory usage: 52.6+ KB


In [18]:

# Round electricity prices to 2 decimal places
price_columns = ["DayAheadPriceEUR", "DayAheadPriceDKK"]

df[price_columns] = df[price_columns].round(2)

# Check the updated prices
df[price_columns].head()


,DayAheadPriceEUR,DayAheadPriceDKK
0,257.39,1923.60
1,257.45,1924.05
2,251.04,1876.15
3,251.12,1876.75
4,239.71,1791.47


In [19]:


# Convert timestamp columns to datetime
df["TimeUTC"] = pd.to_datetime(df["TimeUTC"], utc=True)
df["TimeDK"] = pd.to_datetime(df["TimeDK"])

# Create a date column for easier analysis
df["DeliveryDate"] = df["TimeDK"].dt.date

# Create a folder to store the processed data
output_folder = PROJECT_ROOT / "data" / "processed"
output_folder.mkdir(parents=True, exist_ok=True)

# Keep existing processed data intact when rerunning the notebook.
output_path = output_folder / "day_ahead_prices.parquet"
if output_path.exists():
    print("Existing processed data kept:", output_path)
else:
    with output_path.open("xb") as file:
        df.to_parquet(file, index=False)
    print("Processed data saved successfully!")

# Check the prepared dataset in memory
print("Rows:", len(df))
print("Columns:", len(df.columns))


Existing processed data kept: /Users/yassinehouari/Desktop/Nordic Energy Data Pipeline/data/processed/day_ahead_prices.parquet
Rows: 1344
Columns: 6
